# CodeOrbit agent test with Claude Code - Kaggle

The frontier-model half of the pre-registered test (`bench/PREREGISTRATION.md`):
Claude Code is given each SWE-bench Verified issue twice, without and with
CodeOrbit's MCP tools, and must name the functions to change. The model runs on
Anthropic's side under your Claude plan; the GPU here only computes CodeOrbit's
embeddings, exactly as in the open-model run.

### Once, before the first run
1. On your own computer: `claude setup-token`, and copy the token it prints.
2. Here: **Add-ons -> Secrets -> Add a new secret**, label `CLAUDE_CODE_OAUTH_TOKEN`,
   paste the token, and switch it on for this notebook. Never paste it into a cell.
3. Settings: **Accelerator: GPU T4 x2** (one is used) and **Internet: On**.

### To run
`MODE = 'pilot'` first (3 issues), then `MODE = 'main'` (the 100 pre-registered
issues). **Save Version -> Save & Run All**, then stop the editor session. If your
plan's usage limit is reached, the run stops cleanly; wait for the limit to reset,
add this notebook's output as an input, put the run folder in `RESUME`, and run
again.

## Settings

In [ ]:
MODEL = 'sonnet'   # Claude Code model alias; the exact model ID is recorded in the run
MODE = 'pilot'     # 'pilot' - 3 issues outside the main sample: checks the pipeline
                   # 'main'  - the 100 pre-registered issues: the result to report
MAX_HOURS = 9      # stop cleanly, before your remaining Kaggle quota runs out
RESUME = ''        # to continue: '/kaggle/input/.../codeorbit-claude-runs/<run folder>'

## 1. Get CodeOrbit

In [ ]:
import os
RUNS = '/kaggle/working/codeorbit-claude-runs'
SRC = '/tmp/codeorbit'
os.makedirs(RUNS, exist_ok=True)
%cd /tmp
!rm -rf {SRC}
!git clone -q --depth 1 https://github.com/Anilll-coder/codeorbit.git {SRC}
%cd {SRC}
!git log -1 --format='%h %s'
!pip install -q -e . tokenizers

## 2. Ollama for embeddings only

In [ ]:
!apt-get -qq install -y pciutils lshw zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, requests
subprocess.Popen(['ollama', 'serve'], env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'),
                 stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get('http://127.0.0.1:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull nomic-embed-text

## 3. Claude Code, signed in with your secret

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash > /dev/null
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
from kaggle_secrets import UserSecretsClient
os.environ['CLAUDE_CODE_OAUTH_TOKEN'] = UserSecretsClient().get_secret('CLAUDE_CODE_OAUTH_TOKEN')
!claude --version
# A one-word check that the token works, before any issue is started.
!cd /tmp && claude -p "Reply with the single word OK." --model {MODEL} --tools "" --strict-mcp-config --no-session-persistence --output-format json | python -c "import json,sys; r=json.load(sys.stdin); print('signed in' if not r.get('is_error') else 'NOT signed in: ' + str(r.get('result'))[:200])"

## 4. Download SWE-bench

In [ ]:
!python scripts/swebench.py download --split lite
!python scripts/swebench.py download --split verified

## 5. Run

In [ ]:
import shutil
selection = {
    'pilot': '--split verified --instances psf__requests-1921,pytest-dev__pytest-7205,pytest-dev__pytest-10081',
    'main':  '--split verified --exclude-split lite --sample 100 --seed 3',
}[MODE]
if RESUME:
    dst = os.path.join(RUNS, os.path.basename(RESUME.rstrip('/')))
    if not os.path.exists(dst):
        shutil.copytree(RESUME, dst)                 # /kaggle/input is read-only
    cmd = f'python scripts/claude_eval.py run --resume {dst}'
else:
    cmd = f'python scripts/claude_eval.py run --model {MODEL} {selection} --embed'
cmd += f' --checkouts /tmp/checkouts --out {RUNS} --max-hours {MAX_HOURS}'
print(cmd)
!{cmd}

## 6. Download

In [ ]:
shutil.make_archive('/kaggle/working/codeorbit-claude-runs', 'zip', RUNS)
print('download /kaggle/working/codeorbit-claude-runs.zip from the Output tab')